<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-08-agents-and-adk/lesson-8.2-multi-agent/notebooks/GCP_Capstone_8.2_MultiAgent.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.2 Multi-Agent Orchestration — Routing, a Pipeline, a Fan-Out Across Two Tenants, a Loop That Stops on Evidence
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

Teams of the agent 8.1 built, on the same lane and the same kit. An LLM router that delegates by description, a fixed pipeline that passes state between stages, a parallel fan-out that asks two tenants the same question and shows isolation as two figures, a loop whose stop condition is a tool call, and a workflow wrapped as a tool of the root.


## Setup
Identical to 8.1: the kit cloned and imported, the identity minted per call as `documind-ui-sa`, which sits on the acme and zeta rosters - the fan-out needs both.


In [ ]:
!pip install -q google-adk==2.8.0 google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every agent in this module imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": f"https://documind-api-{NUMBER}.{REGION}.run.app",
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - 8.7's gate fails a paste.
print("kit:", KIT, "| API:", os.environ["RAG_API_URL"])


## Cell 1: The adapters
Two from 8.1, and a third shape: an adapter bound to one tenant at construction, for readers that run in parallel over different corpora.


In [ ]:
from google.adk.tools import ToolContext

# The same two adapters as 8.1. A notebook cannot import another notebook, and the rule is about
# IMPLEMENTATIONS, not adapters: both delegate to the kit's one retrieve() in a single call.

def retrieve(query: str, doc_type: str = "all", top_k: int = 5, tool_context: ToolContext = None) -> dict:
    """Retrieve grounded passages from DocuMind's corpus, with the lane's own cited answer.

    Args:
        query: The question, in natural language.
        doc_type: policy, contract, invoice, report, statute, guidance, form, research_paper, or all.
        top_k: How many passages to return (1-20).
    """
    state = tool_context.state if tool_context is not None else {}
    return documind_tools.retrieve(query, tenant_id=state.get("tenant_id") or TENANT, top_k=max(1, min(int(top_k), 20)),
                                   doc_type=None if doc_type in ("all", "") else doc_type, brain="adk")


def calculate_cost(total_pages: int, processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents.
        processing_type: Service tier - standard, priority, or bulk.
    """
    try:
        return documind_tools.calculate_processing_cost(total_pages, processing_type=processing_type)
    except ValueError as e:
        return {"error": str(e), "valid_tiers": sorted(documind_tools.RATES)}


# A THIRD SHAPE, for the fan-out: an adapter bound to ONE tenant at construction. Two of these run in
# parallel over two corpora inside one session, so the tenant cannot come from shared state - it is
# closed over. The model still never sees it: the schema is query and top_k.
def retrieve_for(tenant: str):
    def retrieve_bound(query: str, top_k: int = 5) -> dict:
        """Retrieve grounded passages from this tenant's DocuMind corpus, with the lane's cited answer.

        Args:
            query: The question, in natural language.
            top_k: How many passages to return (1-20).
        """
        return documind_tools.retrieve(query, tenant_id=tenant, top_k=max(1, min(int(top_k), 20)), brain="adk")
    retrieve_bound.__name__ = f"retrieve_{tenant}"      # the tool's name in the trace: which corpus answered
    return retrieve_bound


print("adapters:", retrieve.__name__, calculate_cost.__name__, retrieve_for("acme").__name__, retrieve_for("zeta").__name__)


## Cell 2: A runner for many roots
Every turn gets a fresh session seeded with the tenant; the trace shows which agent called which tool, and `transfer_to_agent` separately.


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = "gemini-3.6-flash"
session_service = InMemorySessionService()
LAST = {"calls": [], "results": [], "text": [], "transfers": []}


async def run_turn(target, question: str, state: dict | None = None, quiet: bool = False) -> dict:
    """One turn against `target`, in a FRESH session seeded with the tenant. Prints the trace."""
    LAST.update(calls=[], results=[], text=[], transfers=[])
    runner = Runner(agent=target, app_name="documind", session_service=session_service)
    session = await session_service.create_session(app_name="documind", user_id="student",
                                                   state={"tenant_id": TENANT, **(state or {})})
    content = types.Content(role="user", parts=[types.Part.from_text(text=question)])
    async for event in runner.run_async(user_id="student", session_id=session.id, new_message=content):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                name = part.function_call.name
                (LAST["transfers"] if name == "transfer_to_agent" else LAST["calls"]).append(name)
                if not quiet:
                    print(f"  [{event.author}] call {name}({dict(part.function_call.args or {})})")
            if part.function_response:
                LAST["results"].append(part.function_response.response)
                if not quiet:
                    print(f"  [{event.author}] result {str(part.function_response.response)[:140]}")
            if part.text:
                LAST["text"].append((event.author, part.text))
                if not quiet:
                    print(f"  [{event.author}] {part.text.strip()[:300]}")
    s = await session_service.get_session(app_name="documind", user_id="student", session_id=session.id)
    return dict(s.state)


## Cell 3: sub_agents - LLM-driven delegation
The root reads the specialists' *descriptions* and hands off with `transfer_to_agent`. Descriptions are the routing table.


In [ ]:
from google.adk.agents import LlmAgent

CITE = "Answer only from what the tools return and cite the sources they name. If the corpus cannot answer, say so and cite nothing."


def make_specialists():
    """Fresh specialist instances. An ADK agent belongs to ONE parent tree, so every root that
    needs them builds its own copies (or you meet 'already has parent' at construction)."""
    search = LlmAgent(name="SearchAgent", model=MODEL, tools=[retrieve],
                      description="Answers questions about the company's documents: policies, contracts, invoices, statutes.",
                      instruction="You are the document specialist. Call retrieve for every question. " + CITE)
    cost = LlmAgent(name="CostAgent", model=MODEL, tools=[calculate_cost],
                    description="Prices document processing: pages, tiers, rupees.",
                    instruction="You are the pricing specialist. Use calculate_cost; the tiers are standard, priority and bulk.")
    return search, cost


search_agent, cost_agent = make_specialists()
router = LlmAgent(
    name="DocuMind", model=MODEL,
    instruction="Route document questions to SearchAgent and pricing questions to CostAgent. Do not answer document questions yourself.",
    sub_agents=[search_agent, cost_agent],
)

# LLM-DRIVEN DELEGATION: the root reads the sub-agents' DESCRIPTIONS and calls transfer_to_agent.
# The trace shows the hand-off, then the specialist's own tool call. Golden row lk-01: Rs 40,000.
await run_turn(router, "What is the per-trip cap on domestic travel reimbursement?")
print("\ntransfers:", LAST["transfers"], "| tool calls:", LAST["calls"])
assert LAST["transfers"] and "retrieve" in LAST["calls"], "the root did not delegate, or the specialist did not retrieve"


## Cell 4: SequentialAgent - the order is a fact
Each stage writes an `output_key` into state; the next reads it with `{key}` in its instruction. No model decides the order.


In [ ]:
from google.adk.agents import SequentialAgent

# A PIPELINE: fixed order, state as the conveyor belt. Each stage writes its output_key into session
# state and the next stage's instruction reads it with {key}. No LLM decides the order - that is the
# point of SequentialAgent: use it when the order is a fact, not a judgement.
finder = LlmAgent(name="Finder", model=MODEL, tools=[retrieve], output_key="passages",
                  instruction="Call retrieve for the user's question, then output ONLY the quotes it returned, "
                              "one per line, each prefixed with its source_uri and page. No commentary.")
classifier = LlmAgent(name="Classifier", model=MODEL, output_key="classification",
                      instruction="Passages:\n{passages}\n\nClassify: which document type is this (policy, contract, "
                                  "invoice, report, statute, guidance) and which clause or section codes appear? Output two lines.")
answerer = LlmAgent(name="Answerer", model=MODEL, output_key="final",
                    instruction="Passages:\n{passages}\nClassification:\n{classification}\n\nAnswer the user's question "
                                "in two sentences, citing the source and clause code. " + CITE)

pipeline = SequentialAgent(name="AnswerPipeline", sub_agents=[finder, classifier, answerer])

# Golden row lk-12: the ACME MSA's notice for termination for convenience - 90 days, clause MSA-04.
state = await run_turn(pipeline, "What notice does the ACME MSA require to terminate for convenience?", quiet=True)
for key in ("passages", "classification", "final"):
    print(f"\n{key}:\n  " + str(state.get(key, ""))[:400].replace("\n", "\n  "))


## Cell 5: ParallelAgent - two tenants, two figures
The same question to two corpora at once. ACME's handbook caps a trip at Rs 40,000, Zeta's at Rs 25,000: the golden set's isolation row, seen from an agent.


In [ ]:
from google.adk.agents import ParallelAgent

# FAN-OUT ACROSS TWO TENANTS. Two readers run concurrently, each over ITS OWN corpus - the adapter
# is bound to the tenant, so the same question goes to two rosters and two chunk sets - and a
# comparer reads both results from state. This is isolation as two numbers: the golden set's
# sharpest row (iso-01) asks exactly this. ACME's handbook caps a trip at Rs 40,000, Zeta's at
# Rs 25,000, same clause code, different figure.
acme_reader = LlmAgent(name="AcmeReader", model=MODEL, tools=[retrieve_for("acme")], output_key="acme_answer",
                       instruction="Call retrieve_acme for the user's question. Output one line: the figure, the clause code, the source_uri. " + CITE)
zeta_reader = LlmAgent(name="ZetaReader", model=MODEL, tools=[retrieve_for("zeta")], output_key="zeta_answer",
                       instruction="Call retrieve_zeta for the user's question. Output one line: the figure, the clause code, the source_uri. " + CITE)
comparer = LlmAgent(name="Comparer", model=MODEL, output_key="comparison",
                    instruction="ACME: {acme_answer}\nZeta: {zeta_answer}\n\nState both figures with their sources in one sentence each, "
                                "then one sentence on whether they differ.")

comparison = SequentialAgent(name="TwoTenantComparison",
                             sub_agents=[ParallelAgent(name="ReadBoth", sub_agents=[acme_reader, zeta_reader]), comparer])

CAP = "What is the per-trip cap on domestic travel reimbursement?"
state = await run_turn(comparison, CAP, quiet=True)
print("acme :", state.get("acme_answer", "")[:160])
print("zeta :", state.get("zeta_answer", "")[:160])
print("\n" + state.get("comparison", "")[:400])
print("\ntools called:", LAST["calls"])
assert {"retrieve_acme", "retrieve_zeta"} <= set(LAST["calls"]), "both corpora must have been read"


## Cell 6: LoopAgent - stop on evidence
The critic's approval is a tool call, so the stop condition is in the trace; `max_iterations` is the budget behind it.


In [ ]:
from google.adk.agents import LoopAgent
from google.adk.tools import ToolContext

# A LOOP THAT STOPS ON EVIDENCE. The drafter answers; the critic checks for a citation with a clause
# code and calls exit_loop when it finds one, else sends the draft back. The stop condition is a
# TOOL CALL - which makes it visible in the trace - and max_iterations is the budget behind it
# (8.7 names the four kinds of stop condition; this loop has three of them).
def exit_loop(tool_context: ToolContext) -> dict:
    """Call when the draft cites a source and a clause code. Ends the refinement loop."""
    tool_context.actions.escalate = True
    return {"status": "approved"}


drafter = LlmAgent(name="Drafter", model=MODEL, tools=[retrieve], output_key="draft",
                   instruction="Feedback so far (may be empty): {feedback?}\n\nAnswer the user's question from retrieve, "
                               "citing the source_uri and the clause code (like NP-03 or LV-07) of every figure. " + CITE)
critic = LlmAgent(name="Critic", model=MODEL, tools=[exit_loop], output_key="feedback",
                  instruction="Draft:\n{draft}\n\nIf the draft cites at least one clause code AND a source, call exit_loop. "
                              "Otherwise output one line saying what citation is missing.")

refinement = LoopAgent(name="RefinementLoop", sub_agents=[drafter, critic], max_iterations=3)

# Golden row jn-03: two clauses in one answer - encashment (LV-07) and the E3 notice (NP-03).
state = await run_turn(refinement, "I am an E3 leaving with 50 days of earned leave. How much is encashed and what notice do I serve?", quiet=True)
print("draft   :", state.get("draft", "")[:400])
print("feedback:", state.get("feedback", "")[:200])
print("stopped by exit_loop:", "exit_loop" in LAST["calls"], "| retrieve calls:", LAST["calls"].count("retrieve"))


## Cell 7: AgentTool - a workflow as a tool
The two-tenant comparison becomes one tool of the root, beside the specialists it can transfer to.


In [ ]:
from google.adk.tools.agent_tool import AgentTool

# A WORKFLOW AS A TOOL. The comparison from Cell 4 becomes one tool the root can call, beside the
# specialists it can transfer to. Fresh specialists again (the single-parent rule); the comparison
# is wrapped, not re-parented, so the Cell 4 tree is untouched.
fr_search, fr_cost = make_specialists()
compare_tool = AgentTool(agent=comparison)

full_root = LlmAgent(
    name="DocuMind", model=MODEL,
    instruction="Route document questions to SearchAgent and pricing to CostAgent. When the user asks how two "
                "tenants or companies compare, call TwoTenantComparison and relay its result.",
    sub_agents=[fr_search, fr_cost],
    tools=[compare_tool],
)

for q in ["What notice period applies during probation?",                                   # -> SearchAgent (lk-04: 15 days)
          "How do ACME's and Zeta's per-trip travel caps compare?",                          # -> the comparison tool
          "How much would it cost to process 200 pages on the priority tier?"]:              # -> CostAgent ($24.00, Rs 2,040)
    print("\nQ:", q)
    await run_turn(full_root, q, quiet=True)
    print("   transfers:", LAST["transfers"], "| tools:", LAST["calls"])
    print("   A:", (LAST["text"][-1][1] if LAST["text"] else "")[:220].strip())


## Choosing the pattern

| Need | Pattern | The model decides | State |
|---|---|---|---|
| the right specialist for a question | `sub_agents` + `transfer_to_agent` | which agent | shared |
| a fixed sequence of steps | `SequentialAgent` | nothing about order | `output_key` -> `{key}` |
| the same work over several inputs | `ParallelAgent` | nothing | one key per branch |
| retry until a condition holds | `LoopAgent` + an exit tool | when to stop, within a budget | the draft and the feedback |
| a whole workflow inside another | `AgentTool` | whether to call it | the tool's result |

## ✅ Lesson 8.2 complete
- ✅ A router that delegates by description, asserted to have delegated and retrieved
- ✅ A pipeline that passes passages, classification and answer through state
- ✅ A fan-out across acme and zeta: Rs 40,000 and Rs 25,000, two adapters bound to two tenants
- ✅ A loop whose stop is a tool call, on a two-clause question
- ✅ A workflow as a tool of the root
